# Data Preparation Pipeline

This notebook prepares and processes raw data for ASEAN Green Bonds research using the `asean_green_bonds` package.

**Pipeline:**
1. Load raw data (panel, ESG, market, green bonds)
2. Merge datasets with proper identifiers
3. Filter to ASEAN firms and valid years
4. Handle missing values and merge industry data
5. Feature engineering (log transforms, indicators)
6. Save processed data for analysis

In [1]:
# Import ASEAN Green Bonds package
from asean_green_bonds import data, config
import pandas as pd
import numpy as np

print(f'Package imported successfully')
print(f'Data directory: {config.DATA_DIR}')
print(f'Output directory: {config.PROCESSED_DATA_DIR}')

Package imported successfully
Data directory: /Users/bunnypro/Projects/refinitiv-search/data
Output directory: /Users/bunnypro/Projects/refinitiv-search/processed_data


In [2]:
# Load all raw data sources
print('Loading raw data...')

panel_df = data.load_raw_panel_data()
esg_df = data.load_esg_panel_data()
market_df = data.load_market_data()
gb_df = data.load_green_bonds_data(asean_only=True)
series_df = data.load_series_data()

print(f'Panel data: {panel_df.shape}')
print(f'ESG data: {esg_df.shape}')
print(f'Market data: {market_df.shape}')
print(f'Green bonds: {gb_df.shape}')
print(f'Series data: {series_df.shape}')

Loading raw data...
Panel data: (55392, 26)
ESG data: (35240, 10)
Market data: (5126, 4)
Green bonds: (328, 4)
Series data: (4962, 2)


In [3]:
# Merge financial and ESG data
print('Merging datasets...')

merged_df = data.merge_panel_data(panel_df, esg_df, market_df)
print(f'Merged financial + ESG: {merged_df.shape}')

# Add green bonds indicators
merged_df = data.merge_green_bonds(merged_df, gb_df, market_df)
print(f'After green bonds merge: {merged_df.shape}')

# Add industry classification
merged_df = data.merge_industry_data(merged_df, series_df)
print(f'After industry merge: {merged_df.shape}')

Merging datasets...
Merged financial + ESG: (47007, 33)
After green bonds merge: (47007, 40)
After industry merge: (47007, 41)


In [4]:
# Filter to ASEAN firms and valid years
print('Filtering data...')

merged_df = data.filter_asean_firms_and_years(
    merged_df,
    min_year=2015,
    max_year=2024
)
print(f'After ASEAN & year filter: {merged_df.shape}')

# Handle missing values
merged_df = data.handle_missing_values(
    merged_df,
    forward_fill_cols=[
        'total_assets', 'total_debt', 'long_term_debt',
        'market_capitalization', 'employees'
    ],
    min_years_per_firm=3
)
print(f'After missing value handling: {merged_df.shape}')

Filtering data...
After ASEAN & year filter: (42250, 41)
After missing value handling: (42102, 41)


In [5]:
# Create log-transformed features
print('Engineering features...')

merged_df = data.create_log_features(
    merged_df,
    cols_to_log=['total_assets', 'employees', 'net_sales_or_revenues']
)
print(f'Log features created')

# Normalize percentages
merged_df = data.normalize_percentages(
    merged_df,
    pct_cols=['return_on_assets', 'return_on_equity_total']
)
print(f'Percentages normalized')

# Winsorize outliers
merged_df = data.winsorize_outliers(merged_df)
print(f'Outliers winsorized')

# Encode categorical features
merged_df = data.encode_categorical_features(merged_df)
print(f'Categorical features encoded')

Engineering features...
Log features created
Percentages normalized
Outliers winsorized
Categorical features encoded


In [6]:
# Summary statistics
print('\n' + '='*60)
print('PROCESSING SUMMARY')
print('='*60)
print(f'Final dataset shape: {merged_df.shape}')
print(f'Entities (firms): {merged_df["ric"].nunique()}')
print(f'Time periods: {merged_df["Year"].nunique()}')
print(f'Green bond issuers: {(merged_df["green_bond_issue"] == 1).sum()}')
print(f'Certified bonds: {(merged_df["is_certified"] == 1).sum()}')
print(f'\nMissing data:')
print(merged_df.isnull().sum().sum(), 'cells missing')
print(f'Missing rate: {merged_df.isnull().sum().sum() / (merged_df.shape[0] * merged_df.shape[1]) * 100:.2f}%')


PROCESSING SUMMARY
Final dataset shape: (42102, 44)
Entities (firms): 4593
Time periods: 10
Green bond issuers: 34
Certified bonds: 0

Missing data:
502173 cells missing
Missing rate: 27.11%
